# Exercises XP Gold - RNNs in PyTorch on MNIST CSV

This is a guided notebook for the exercise on the platform. Cells marked **PREFILLED** are for execution only. Cells marked **To-Do** require your action. When a written answer is required, the **To-Do** appears inside a markdown cell. When code is required, the **To-Do** appears inside a code cell as comments.

Learning points appear for key ideas.

## What you will learn
- Implement and train RNN, GRU, and LSTM with PyTorch
- Preprocess and load sequence datasets
- Evaluate models with accuracy
- Use Dataset and DataLoader for efficient IO

## What you will create
- A custom dataset class for MNIST CSV
- Three sequence models: SimpleRNN, SimpleGRU, SimpleLSTM
- A training loop and an evaluation function

## How I run this notebook
- Open the file in Google Colab (File → Upload notebook).
- Run all cells top to bottom (Runtime → Run all). The notebook downloads the MNIST CSV zip from the course GitHub itself, so no local files are needed.
- Seeds are fixed (`SEED = 42`), so the numbers should be close to mine.
- **CPU run (what I used here):** `EPOCHS = 2` (fewer epochs, so the numbers are lower than a full run). The datasets cell switches to this automatically when no GPU is found.
- **Colab GPU:** the template settings apply: `EPOCHS = 5`. The full 60,000-row training set is used on both CPU and GPU.
- DataLoaders use `num_workers=0`, so the notebook also runs on Windows.

# Part 1: Setting up the environment

**As stated in the exercise**  
Install libraries, define hyperparameters, and set the device.

**PREFILLED**  
Imports, seed control, hyperparameters, device selection, and simple helpers.

**What this cell does:** imports the libraries, fixes the random seed, sets the hyperparameters, and picks the device (GPU if available, otherwise CPU).

In [1]:
# PREFILLED: just execute
import os, time, math, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Hyperparameters - you may adjust in the To-Do cell below
INPUT_SIZE   = 28   # number of features per time step for MNIST row-wise
SEQ_LEN      = 28   # number of time steps
HIDDEN_SIZE  = 128
NUM_LAYERS   = 1
NUM_CLASSES  = 10
LR           = 1e-3
EPOCHS       = 5
BATCH_SIZE   = 128

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

**To-Do (written):** Briefly justify the choice of treating a 28x28 image as a sequence of length 28 with input size 28. When would column-wise sequencing change anything in practice?

**My answer:** I read each 28x28 image as 28 time steps, one image row per step. Each step is a vector of 28 pixel values, so the input size is 28. I chose rows because each row is a short horizontal slice of the digit. The network therefore reads the digit from top to bottom, and its last hidden state summarizes the whole picture. The CSV header names the pixels row by row (`1x1` … `1x28` is the first row), so `reshape(28, 28)` with no transpose gives exactly this layout.

Column-wise order changes what each step sees: every step becomes a vertical slice, and the last state summarizes the image from left to right. That matters when the scan direction carries meaning, for example stroke order in handwriting, or when the model only sees part of the sequence. For a plain MNIST classifier I would expect only small accuracy differences. I have not measured that here.

**Learning point**  
A grayscale image can be read as a sequence by rows or columns. The model consumes vectors of size 28 across 28 steps. Information flow differs slightly by ordering, but both work because local spatial patterns still get exposed to the recurrent state.

# Part 2: Designing the neural network models

**As stated in the exercise**  
Implement SimpleRNN, SimpleGRU, and SimpleLSTM. Each ends with a fully connected layer mapping the last hidden state to 10 classes.

**PREFILLED**  
Model skeletons with clear To-Do regions.

**What this cell does:** defines the three models. Each one runs its recurrent layer over the 28 rows and maps the last hidden state to 10 logits (one score per digit).

In [2]:
# PREFILLED: just execute
class SimpleRNNModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super().__init__()
        # batch_first=True means inputs are [batch, seq_len, input_size]
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc  = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # x shape: [batch, seq_len, input_size]
        _, h_n = self.rnn(x)          # h_n: [num_layers, batch, hidden]
        last = h_n[-1]                # last layer's hidden state after the final row: [batch, hidden]
        return self.fc(last)          # logits [batch, num_classes]


class SimpleGRUModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True)
        self.fc  = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        _, h_n = self.gru(x)          # h_n: [num_layers, batch, hidden]
        last = h_n[-1]                # [batch, hidden]
        return self.fc(last)          # logits [batch, num_classes]


class SimpleLSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc   = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        _, (h_n, _c_n) = self.lstm(x) # LSTM returns (h_n, c_n); we only need h_n
        last = h_n[-1]                # [batch, hidden]
        return self.fc(last)          # logits [batch, num_classes]

**Learning point**  
RNN keeps one hidden. GRU adds gates that control update and reset. LSTM keeps a cell state with input, forget, and output gates. These gates help retain or discard information over time steps.

# Part 3: Creating a custom dataset class

**As stated in the exercise**  
Create `MnistCsvDataset` that loads the CSV, preprocesses, and yields items compatible with sequences.

Dataset format expectation: a CSV with `label` as the first column, followed by 784 pixel columns named `pixel0 ... pixel783` or unnamed numeric columns. If your CSV differs, adapt the loader.

**Note:** the course CSV is a bit different. Its header is `label,1x1,1x2,...,28x28`, so I select the columns by position (label first, then 784 pixels) instead of by name.

**What this cell does:** defines `MnistCsvDataset`. It reads the CSV, takes the label from column 0 and the 784 pixels from the rest, scales them to [0, 1], and reshapes each row to a 28x28 image.

In [3]:
# PREFILLED: just execute (To-Do filled in)
class MnistCsvDataset(Dataset):
    def __init__(self, csv_path):
        super().__init__()
        df = pd.read_csv(csv_path)
        # Labels: first column, as int64 (CrossEntropyLoss needs integer class ids)
        self.y = df.iloc[:, 0].to_numpy(dtype=np.int64)
        # Pixels: the other 784 columns (header 1x1..28x28), scaled from 0..255 to [0, 1]
        pixels = df.iloc[:, 1:].to_numpy(dtype=np.float32) / 255.0   # [N, 784]
        assert pixels.shape[1] == 784, f"expected 784 pixel columns, got {pixels.shape[1]}"
        # Reshape to [N, 28, 28]. The CSV is row-major, so no transpose: row = time step.
        self.X = pixels.reshape(-1, 28, 28)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        seq = torch.from_numpy(self.X[idx])          # FloatTensor [28, 28] (float32)
        label = torch.as_tensor(self.y[idx], dtype=torch.long)  # LongTensor scalar
        return seq, label

**What this cell does:** downloads the course zip from GitHub and unpacks the train and test CSVs into a `data/` folder next to the notebook. Nothing has to be uploaded by hand.

In [4]:
# What this cell does: downloads and unpacks the MNIST CSV files (works on Colab, Windows, and Mac)
import urllib.request, zipfile

DATA_URL = "https://github.com/devtlv/Datasets-GEN-AI-Bootcamp/raw/refs/heads/main/Week%206/W6D4/MNIST%20in%20CSV.zip"
ROOT = Path("data")                       # on Colab this is /content/data
ROOT.mkdir(parents=True, exist_ok=True)

outer_zip = ROOT / "MNIST in CSV.zip"
if not outer_zip.exists():                # skip the download if the cell is re-run
    urllib.request.urlretrieve(DATA_URL, outer_zip)
with zipfile.ZipFile(outer_zip) as z:
    z.extractall(ROOT)

DATA_DIR = ROOT / "MNIST in CSV"          # the outer zip holds two inner zips
for split in ["mnist_train", "mnist_test"]:
    if not (DATA_DIR / f"{split}.csv").exists():
        with zipfile.ZipFile(DATA_DIR / f"{split}.zip") as z:
            z.extractall(DATA_DIR)        # each inner zip holds <split>.csv
print(sorted(p.name for p in DATA_DIR.glob("*.csv")))

['mnist_test.csv', 'mnist_train.csv']


**To-Do (code):** Instantiate training and test datasets from CSV files. Wrap them in DataLoaders.

Use variables below. `DATA_DIR` comes from the download cell above, so no paths need to be typed by hand.

**What this cell does:** builds the train and test datasets and their DataLoaders. If there is no GPU, it switches to the CPU setting of 2 epochs.

In [5]:
# To-Do (filled in): create datasets and loaders
TRAIN_CSV = DATA_DIR / "mnist_train.csv"
TEST_CSV  = DATA_DIR / "mnist_test.csv"
ds_train = MnistCsvDataset(TRAIN_CSV)
ds_test  = MnistCsvDataset(TEST_CSV)

# CPU shortcut (documented): without a GPU, train for only 2 epochs so the notebook finishes in minutes.
# The full 60,000-row training set is still used. On a Colab GPU this block is skipped, so EPOCHS=5.
if not torch.cuda.is_available():
    EPOCHS = 2

# num_workers=0 works the same on Windows, Mac, and Colab
dl_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=0, pin_memory=torch.cuda.is_available())
dl_test  = DataLoader(ds_test,  batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=0, pin_memory=torch.cuda.is_available())
print(f"device={device} | epochs={EPOCHS} | train_rows={len(ds_train)} | test_rows={len(ds_test)}")

device=cpu | epochs=2 | train_rows=60000 | test_rows=10000


# Part 4: Training the model

**As stated in the exercise**  
Use CrossEntropyLoss and Adam. Implement a training loop that prints epoch loss.

**What this cell does:** defines `train_one_epoch` (one pass over the training data) and `fit` (runs several epochs with Adam and cross-entropy loss).

In [6]:
# PREFILLED: training helpers (To-Do filled in)
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    total_samples = 0    # changed from total_batches: we average per sample, like evaluate_loss
    for xb, yb in loader:
        xb = xb.to(device)                       # [B, SEQ_LEN, INPUT_SIZE]
        yb = yb.to(device)                       # [B]
        assert xb.shape[1:] == (SEQ_LEN, INPUT_SIZE)
        optimizer.zero_grad()                    # clear gradients from the previous batch
        logits = model(xb)                       # forward pass -> [B, 10]
        loss = criterion(logits, yb)             # mean loss over this batch
        loss.backward()                          # compute gradients
        optimizer.step()                         # update the weights
        # weight by batch size so the epoch average is per sample (not per batch)
        total_loss += loss.item() * xb.size(0)
        total_samples += xb.size(0)
    return total_loss / max(1, total_samples)

def fit(model, dl_train, dl_val, epochs, device):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    best_val = math.inf
    for epoch in range(1, epochs+1):
        t0 = time.time()
        train_loss = train_one_epoch(model, dl_train, criterion, optimizer, device)
        val_loss = evaluate_loss(model, dl_val, criterion, device)
        dt = time.time() - t0
        print(f"epoch {epoch:02d} - train_loss {train_loss:.4f} - val_loss {val_loss:.4f} - time_s {dt:.1f}")
        if val_loss < best_val:
            best_val = val_loss
    return model

**To-Do (code):** Implement `evaluate_loss` to compute average loss on a loader. Then construct one of the models, send to device, and call `fit`.

**What this cell does:** defines `evaluate_loss` (average loss with no gradient updates) and trains the RNN for `EPOCHS` epochs.

In [7]:
# To-Do (filled in): evaluation and training
def evaluate_loss(model, loader, criterion, device):
    model.eval()
    total = 0.0; n = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            total += loss.item() * xb.size(0)
            n += xb.size(0)
    return total / max(1, n)

rnn_model = SimpleRNNModel(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, NUM_CLASSES).to(device)
# fit prints a "val_loss" computed on the test loader. It is only printed, never used to pick a model.
rnn_model = fit(rnn_model, dl_train, dl_test, EPOCHS, device)

epoch 01 - train_loss 0.8976 - val_loss 0.4747 - time_s 3.8


epoch 02 - train_loss 0.3977 - val_loss 0.2911 - time_s 3.6


# Part 5: Evaluating the model

**As stated in the exercise**  
Implement accuracy for train and test. Use the trained model to predict test labels and compute accuracy.

**What this cell does:** defines `accuracy` (share of correct predictions) and prints the RNN's train and test accuracy.

In [8]:
# To-Do (filled in): accuracy function
def accuracy(model, loader, device):
    model.eval()
    correct = 0; total = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device); yb = yb.to(device)
            logits = model(xb)
            preds = logits.argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)
    return correct / max(1, total)

print({'train_acc': accuracy(rnn_model, dl_train, device), 'test_acc': accuracy(rnn_model, dl_test, device)})

{'train_acc': 0.9146666666666666, 'test_acc': 0.916}


**To-Do (code):** Repeat training and evaluation for `SimpleGRUModel` and `SimpleLSTMModel`. Compare the final test accuracies.

**What this cell does:** trains the GRU and the LSTM with the same recipe as the RNN (same seed, same settings). Then it prints train and test accuracy for all three models in one table.

In [9]:
# GRU and LSTM: same recipe as the RNN, then one table with train/test accuracy for all three
results = {"RNN": rnn_model}              # the RNN was trained in the previous code cell
for name, cls in [("GRU", SimpleGRUModel), ("LSTM", SimpleLSTMModel)]:
    torch.manual_seed(SEED)               # same starting seed for a fair comparison
    model = cls(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, NUM_CLASSES).to(device)
    results[name] = fit(model, dl_train, dl_test, EPOCHS, device)

print("model  train_acc  test_acc")
for name, model in results.items():
    tr = accuracy(model, dl_train, device)
    te = accuracy(model, dl_test, device)
    print(f"{name:5s}  {tr:.4f}     {te:.4f}")

epoch 01 - train_loss 0.7174 - val_loss 0.2898 - time_s 8.6


epoch 02 - train_loss 0.2162 - val_loss 0.1558 - time_s 8.3


epoch 01 - train_loss 0.6587 - val_loss 0.2371 - time_s 5.8


epoch 02 - train_loss 0.2039 - val_loss 0.1514 - time_s 4.7
model  train_acc  test_acc


RNN    0.9147     0.9160


GRU    0.9552     0.9554


LSTM   0.9580     0.9548


**My answer:** On my CPU run (`EPOCHS = 2`, full training set, seed 42) the test accuracies were RNN 91.6%, GRU 95.5%, and LSTM 95.5% (0.9548 vs 0.9554). Both gated models beat the plain RNN by about 4 points. That fits the idea that gates help the model keep information across 28 steps. GRU and LSTM are effectively tied. A 0.06-point gap from a single seed is noise, so I would not call a winner between them.

The RNN's train (91.5%) and test (91.6%) accuracies are almost equal, so after 2 epochs it is underfitting rather than overfitting. Longer training (`EPOCHS = 5` on a Colab GPU) will probably raise all three. I have not run that here.